In [1]:
# ============================================================
# Class-weighted ML comparison using Atom Pair fingerprints
# Stratified 10-fold CV with identical folds
# ============================================================

import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, precision_recall_curve, auc,
    matthews_corrcoef
)
from sklearn.utils.class_weight import compute_class_weight

from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    ExtraTreesClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import AdaBoostClassifier
#from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import average_precision_score

import os
import joblib

# Directory to save models
MODEL_DIR = "saved_models/chembl_topo"
os.makedirs(MODEL_DIR, exist_ok=True)


# ------------------------------------------------------------
# 1. Load dataset
# ------------------------------------------------------------
df = pd.read_csv("dataset/final_standard_smiles.csv")

df["mol"] = df["standard_smiles"].apply(Chem.MolFromSmiles)
df = df[df["mol"].notnull()].copy()

# Generate Topological Torsion fingerprints (default nBits=2048)
def mol_to_tt_fp(mol):
    return rdMolDescriptors.GetHashedTopologicalTorsionFingerprintAsBitVect(mol, nBits=2048)

df['fp'] = df['mol'].apply(mol_to_tt_fp)

# ✅ Convert fingerprints to NumPy arrays
X = np.array([np.array(fp) for fp in df['fp']])
y = df['class'].astype(int)



# ------------------------------------------------------------
# 3. Class weights (prioritise Active = 1)
# ------------------------------------------------------------
classes = np.unique(y)
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y
)
class_weight_dict = dict(zip(classes, class_weights))

print("Class weights:", class_weight_dict)

# ------------------------------------------------------------
# 4. Stratified 10-fold (IDENTICAL splits)
# ------------------------------------------------------------
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
splits = list(skf.split(X, y))

# ------------------------------------------------------------
# 5. Models (class-weighted where supported)
# ------------------------------------------------------------
models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight=class_weight_dict
    ),
    "Extra Trees": ExtraTreesClassifier(
        random_state=42,
        class_weight=class_weight_dict
    ),
    "Decision Tree": DecisionTreeClassifier(
        random_state=42,
        class_weight=class_weight_dict
    ),
    "SVM": SVC(
        kernel="rbf",
        probability=True,
        class_weight=class_weight_dict
    ),
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight=class_weight_dict
    ),
    "AdaBoost": AdaBoostClassifier(
        n_estimators=300,
        learning_rate=0.5,
        random_state=42
    ),
    "LightGBM": LGBMClassifier(
        class_weight=class_weight_dict
    ),
    "Gradient Boosting": GradientBoostingClassifier(),
    "KNN": KNeighborsClassifier(),
    "Naive Bayes": GaussianNB()
}

def enrichment_factor(y_true, y_scores, top_percent=0.01):
    """
    Compute Enrichment Factor at top_percent (e.g., 0.01 for 1%)
    """
    # Sort by predicted score (descending)
    sorted_idx = np.argsort(y_scores)[::-1]
    y_true_sorted = np.array(y_true)[sorted_idx]

    n_total = len(y_true)
    n_actives = np.sum(y_true)

    top_n = int(np.ceil(top_percent * n_total))
    top_hits = y_true_sorted[:top_n]

    n_top_actives = np.sum(top_hits)

    if n_actives == 0:
        return 0

    ef = (n_top_actives / top_n) / (n_actives / n_total)
    return ef




def bedroc_score(y_true, y_scores, alpha=20.0):
    y_true = np.array(y_true)
    y_scores = np.array(y_scores)

    # Sort by predicted score
    order = np.argsort(y_scores)[::-1]
    y_true = y_true[order]

    n = len(y_true)
    n_pos = np.sum(y_true)

    if n_pos == 0:
        return 0.0

    # Positions of actives (1-based)
    ranks = np.where(y_true == 1)[0] + 1

    # Relative positions
    ri = ranks / n

    # RIE (Robust Initial Enhancement)
    rie = np.sum(np.exp(-alpha * ri)) / n_pos

    # Normalization constants
    rand = (1 - np.exp(-alpha)) / (alpha)
    max_rie = (1 - np.exp(-alpha * (n_pos / n))) / (alpha * (n_pos / n))

    bedroc = (rie - rand) / (max_rie - rand)

    return bedroc


# ------------------------------------------------------------
# 6. Cross-validation
# ------------------------------------------------------------
results = []

for name, model in models.items():
    print(f"\n🔍 Evaluating {name}")

    accs, precs, recs, f1s = [], [], [], []
    rocs, pras, mccs = [], [], []
    ef1s, ef5s, ef10s, bedrocs = [], [], [], []

    for fold, (train_idx, test_idx) in enumerate(splits, 1):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        if hasattr(model, "predict_proba"):
            y_prob = model.predict_proba(X_test)[:, 1]
        else:
            y_prob = None

        accs.append(accuracy_score(y_test, y_pred))
        precs.append(precision_score(y_test, y_pred, zero_division=0))
        recs.append(recall_score(y_test, y_pred, zero_division=0))
        f1s.append(f1_score(y_test, y_pred, zero_division=0))
        mccs.append(matthews_corrcoef(y_test, y_pred))

        if y_prob is not None:
            rocs.append(roc_auc_score(y_test, y_prob))

            pras.append(average_precision_score(y_test, y_prob))

            ef1s.append(enrichment_factor(y_test, y_prob, 0.01))
            ef5s.append(enrichment_factor(y_test, y_prob, 0.05))
            ef10s.append(enrichment_factor(y_test, y_prob, 0.10))

            bedrocs.append(bedroc_score(y_test, y_prob, alpha=20.0))

    results.append([
        name,
        np.mean(accs), np.std(accs),
        np.mean(precs), np.std(precs),
        np.mean(recs), np.std(recs),
        np.mean(f1s), np.std(f1s),
        np.mean(rocs), np.std(rocs),
        np.mean(pras), np.std(pras),
        np.mean(mccs), np.std(mccs),
        np.mean(ef1s), np.std(ef1s),
        np.mean(ef5s), np.std(ef5s),
        np.mean(ef10s), np.std(ef10s),
        np.mean(bedrocs), np.std(bedrocs)
    ])


# ------------------------------------------------------------
# 7. Results table
# ------------------------------------------------------------
columns = [
    "Model",
    "Accuracy_mean", "Accuracy_std",
    "Precision_mean", "Precision_std",
    "Recall_mean", "Recall_std",
    "F1_mean", "F1_std",
    "ROC_AUC_mean", "ROC_AUC_std",
    "PR_AUC_mean", "PR_AUC_std",
    "MCC_mean", "MCC_std",
    "EF1_mean", "EF1_std",
    "EF5_mean", "EF5_std",
    "EF10_mean", "EF10_std",
    "BEDROC_mean", "BEDROC_std"
]

results_df = pd.DataFrame(results, columns=columns)

# Sort by early-recognition performance
results_df = results_df.sort_values(
    ["BEDROC_mean", "EF1_mean", "PR_AUC_mean"],
    ascending=False
)

print("\n==================== 10-FOLD CV RESULTS ====================")
print(results_df)

results_df.to_csv(
    "result/chembl_topo_classweighted_10fold_metrics.csv",
    index=False
)

Class weights: {0: 1.1481113320079523, 1: 0.8857361963190185}

🔍 Evaluating Random Forest

🔍 Evaluating Extra Trees

🔍 Evaluating Decision Tree

🔍 Evaluating SVM

🔍 Evaluating Logistic Regression

🔍 Evaluating AdaBoost

🔍 Evaluating LightGBM
[LightGBM] [Info] Number of positive: 2347, number of negative: 1811
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008047 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1576
[LightGBM] [Info] Number of data points in the train set: 4158, number of used features: 788
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.499951 -> initscore=-0.000196
[LightGBM] [Info] Start training from score -0.000196
[LightGBM] [Info] Number of positive: 2347, number of negative: 1811
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.010238 seconds.
You can set `force_row_wise=t